In [9]:
import random
import torch
import torch.nn as nn

1. Encoder: 입력 시퀀스를 읽어 Context Vector를 생성

In [4]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        # src Shape: [batch_size, src_len]
        embedded = self.embedding(src)
        # -> [batch_size, src_len, emb_dim]

        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs Shape: [batch_size, src_len, hidden_di,]
        # hidden Shape: [1, batch_size, hidden_dim]

        return hidden, cell

In [5]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_step, hidden, cell):
        # input_step Shape: [batch_size]
        input_step = input_step.unsqueeze(1) # 데이터를 모델의 규격에 맞게 tensor 차원을 N차원 늘려주는 함수
        # input_step Shape: [batch_size, 1]

        embedded = self.embedding(input_step)

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell)) # 초기 상태값 입력
        # output Shape: [batch_size, 1, hidden_dim]

        prediction = self.fc_out(output.squeeze(1)) # 차원 축소
        # prediction Shape : [batch_size, output_dim]

        return prediction, hidden, cell
    

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src Shape: [batch_size, src_len]
        batch_size = src.shape[0]

        # trg Shape: [batch_size, trg_len]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        # 디코더의 예측 결과를 저장할 텐서 초기화
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        # 첫 번째 입력값은 디코더의 초기 입력값으로 사용
        input = trg[:, 0]

        # 타겟 시퀀스 길이 만큼 반복하며 디코더 실행
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            # Teacher Forcing 적용 여부
            teacher_force = random.random() < teacher_forcing_ratio

            # Teacher Forcing이 적용되지 않은 경우
            top1 = output.argmax(1)
            
            # 확률에 따라 정답 토큰(Ground Truth) 또는 모델 예측 값을 다음 입력으로 전달
            decoder_input = trg[:, t] if teacher_force else top1

        return outputs


In [10]:
# 하이퍼파라미터 설정
INPUT_DIM = 1000   # 입력 언어 단어장 크기 (Vocabulary Size)
OUTPUT_DIM = 1200  # 출력 언어 단어장 크기
EMB_DIM = 64       # 임베딩 차원
HIDDEN_DIM = 128   # LSTM 은닉 상태 차원
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 인스턴스 생성
enc = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM)
model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)

# 가상의 배치 데이터 생성 (Batch Size: 4, 입력길이: 7, 출력길이: 9)
# 0번 토큰이 <SOS> 역할을 한다고 가정
dummy_src = torch.randint(0, INPUT_DIM, (4, 7)).to(DEVICE)
dummy_trg = torch.randint(0, OUTPUT_DIM, (4, 9)).to(DEVICE)

# 모델 추론
output = model(dummy_src, dummy_trg, teacher_forcing_ratio=0.5)

print(f"입력 Tensor Shape  (src): {dummy_src.shape}")
print(f"타겟 Tensor Shape  (trg): {dummy_trg.shape}")
print(f"출력 Tensor Shape (out): {output.shape}")

NameError: name 'top1' is not defined